# Research Question 2: Intra-Team Performance Dynamics
**ST312 Project - Candidates: 60726, 61881**

# Exploration

In [11]:
import fastf1
import numpy as np
import pandas as pd
import logging

logging.getLogger('fastf1').setLevel(logging.ERROR)
fastf1.Cache.enable_cache("f1_cache")

In [12]:
bahrain_race_2023 = fastf1.get_session(2023, 'Bahrain', 'Race')
bahrain_race_2023.load()

In [13]:
bahrain_race_2023.laps.columns

Index(['Time', 'Driver', 'DriverNumber', 'LapTime', 'LapNumber', 'Stint',
       'PitOutTime', 'PitInTime', 'Sector1Time', 'Sector2Time', 'Sector3Time',
       'Sector1SessionTime', 'Sector2SessionTime', 'Sector3SessionTime',
       'SpeedI1', 'SpeedI2', 'SpeedFL', 'SpeedST', 'IsPersonalBest',
       'Compound', 'TyreLife', 'FreshTyre', 'Team', 'LapStartTime',
       'LapStartDate', 'TrackStatus', 'Position', 'Deleted', 'DeletedReason',
       'FastF1Generated', 'IsAccurate'],
      dtype='object')

In [14]:
bahrain_race_2023.laps.iloc[0]

Time                      0 days 01:04:15.902000
Driver                                       VER
DriverNumber                                   1
LapTime                   0 days 00:01:39.019000
LapNumber                                    1.0
Stint                                        1.0
PitOutTime                                   NaT
PitInTime                                    NaT
Sector1Time                                  NaT
Sector2Time               0 days 00:00:42.414000
Sector3Time               0 days 00:00:23.842000
Sector1SessionTime                           NaT
Sector2SessionTime        0 days 01:03:52.119000
Sector3SessionTime        0 days 01:04:16.010000
SpeedI1                                    232.0
SpeedI2                                    231.0
SpeedFL                                    278.0
SpeedST                                    252.0
IsPersonalBest                             False
Compound                                    SOFT
TyreLife            

# Columns
| Column | Description |
|--------|-------------|
| Time | Session time when the lap was set (end of lap) |
| Driver | Three-letter drive identifier |
| DriverNumber | Driver Number |
| LapTime | Recorded lap time (timedelta) |
| LapNumber | Recorded lap number |
| Stint | Stint number (increments each time driver pits) |
| PitOutTime | Session time when car exited the pit |
| PitInTime | Session time when car entered the pit |
| Sector1Time | Sector 1 recorded time |
| Sector2Time | Sector 2 reocrded time |
| Sector3Time | Sector 3 recorded time |
| Sector1SessionTime | Session time when sector 1 time was set |
| Sector2SessionTime | Session time when sector 2 time was set |
| Sector3SessionTime | Session time when sector 3 time was set |
| SpeedI1 | Speed trap in sector 1 (km/h) |
| SpeedI2 | Speed trap in sector 2 (km/h) |
| SpeedFL | Speed trap at finish line (km/h) |
| SpeedST | Speed trap on longest straight (km/h) |
| IsPersonalBest | Whether this is the driver's orfficial personal best lap (track-limit violations excluded) |
| Compound | Tyre compound: SOFT, MEDIUM, HARD, INTERMEDIATE, WET, or UNKNOWN |
| TyreLife | Laps driven on this tyre (includes laps from other sesions if used set) |
| FreshTyre | True if tyre at TyreLife=0 at stint start |
| Team | Team name |
| LapStartTime | Session time at the start of the lap |
| LapStartDate | Timestamp at the start of the lap|
| TrackStatus | String of status codes active during the lap ("1" = green, "4" = saftey car, etc)|
| Position | Driver's position at the end of each lap  NaN for crashes |
| Deleted | Whether the lap was deleted by stewards |
| DeletedReason | Reason for lap deletion |
| FastF1Generated | Whether FastF! synthetically added this lap |
| IsAccurate | Whether start/end times are synced correctly with other laps |

# Filtering

We use the following criteria:
- `IsAccurate = True`
- `Deleted != True`
- `FastF1Generated != True`
- `TrackStatus = "1"`
- Exclude lap 1 (it's a standing start, not representative)
- `PitInTime = NA` and `PitOutTime = NA` -> For lap time comparison, we don't want to include pits because drivers artificially slow down during pits

In [15]:
laps = bahrain_race_2023.laps

print('total laps: ', bahrain_race_2023.total_laps)
print("total rows: ", laps.shape[0])
print('rows with IsAccurate == False', len(laps[~laps.IsAccurate]))
print('rows with Deleted == True', len(laps[laps.Deleted]))
print('rows with FastF1Generated == True', len(laps[laps.FastF1Generated]))
print('rows with TrackStatus != 1: ', len(laps[laps.TrackStatus != "1"]))
print('rows of the first lap: ', len(laps[laps.LapNumber == 1]))
print('non-null pit in and pit out times: ', len(laps[laps.PitInTime.notna() | laps.PitOutTime.notna()]))
clean = laps[
      (laps.IsAccurate == True) &
      (laps.Deleted != True) &
      (laps.FastF1Generated != True) &
      (laps.TrackStatus == "1") &
      (laps.LapNumber != 1) &
      (laps.PitInTime.isna()) &
      (laps.PitOutTime.isna())
  ]
print('clean rows:', len(clean))

total laps:  57
total rows:  1056
rows with IsAccurate == False 142
rows with Deleted == True 17
rows with FastF1Generated == True 1
rows with TrackStatus != 1:  76
rows of the first lap:  20
non-null pit in and pit out times:  102
clean rows: 872


# Do the drivers have roughtly equal number of laps?

In [16]:
clean.groupby(['Team', 'Driver']).LapTime.count().sort_index()

Team             Driver
Alfa Romeo       BOT       48
                 ZHO       46
AlphaTauri       DEV       46
                 TSU       49
Alpine           GAS       47
                 OCO       30
Aston Martin     ALO       49
                 STR       49
Ferrari          LEC       32
                 SAI       49
Haas F1 Team     HUL       43
                 MAG       47
McLaren          NOR       36
                 PIA       10
Mercedes         HAM       48
                 RUS       48
Red Bull Racing  PER       49
                 VER       49
Williams         ALB       49
                 SAR       48
Name: LapTime, dtype: int64

Since some drivers (Piastri, Norris) have such low lap numbers, they might've retired early. Let's check what happened

In [17]:
results = bahrain_race_2023.results
results[results.Abbreviation.isin(['PIA', 'OCO', 'LEC', 'NOR'])][["Abbreviation", "Status", 'Laps', 'Position']]

,Abbreviation,Status,Laps,Position
4,NOR,Lapped,55.0,17.0
31,OCO,Retired,41.0,18.0
16,LEC,Retired,39.0,19.0
81,PIA,Retired,13.0,20.0


We come up with a rule:
If the total number of laps between the driver is too big, then we exclude that pair from lap-time analysis

Since the there are two regulation eras (2018-2021 and 2022-2024), it is a confounder. Smiilarly team mates change per season as well. So we need to divide the data in the following hierarchy:
- Regulation Era
    - Session (Year)
        - Race
            - Team
                - Teammate Pair (Driver A - Driver B)
                    - Individual laps

# Getting the data

In [ ]:
import time


def clean_data(laps_df):
    return laps_df[
        (laps_df.IsAccurate == True)
        & (laps_df.Deleted != True)
        & (laps_df.FastF1Generated != True)
        & (laps_df.TrackStatus == "1")
        & (laps_df.LapNumber != 1)
        & (laps_df.PitInTime.isna())
        & (laps_df.PitOutTime.isna())
    ]


def get_race_data(start_year, end_year):
    summary = []
    for year in range(start_year, end_year):
        try:
            time.sleep(5)
            schedule = fastf1.get_event_schedule(year)
            print(f"{year}: {len(schedule)} events")
            races = schedule[schedule.EventFormat != "testing"]['RoundNumber'].values
            for race in races:
                try:
                    time.sleep(5)
                    race_data = fastf1.get_session(year, race, "Race")
                    race_data.load()
                    if not hasattr(race_data, '_laps'):
                        print(f"skipping {year} round {race}: no lap data")
                        continue
                except Exception as e:
                    print(f"Skipping {year} round {race}: {e}")
                    continue
                clean = clean_data(race_data.laps)
                teams = clean.groupby('Team')['Driver'].unique()
                for team, drivers in teams.items():
                    if len(drivers) != 2:
                        continue
                    d1_count = len(clean[clean.Driver == drivers[0]])
                    d2_count = len(clean[clean.Driver == drivers[1]])
                    ratio = round(min(d1_count, d2_count) / max(d1_count, d2_count), 2)

                    summary.append({
                        "year": year,
                        "race": race,
                        "team": team,
                        "driver1": drivers[0],
                        "driver2": drivers[1],
                        "d1_laps": d1_count,
                        "d2_laps": d2_count,
                        "ratio": ratio,
                        "valid": ratio >= 0.75
                })
        except Exception as e:
            print(f"{year}: FAILED {e}")

    summary_df = pd.DataFrame(summary)
    return summary_df

In [19]:
summary_df = get_race_data(2023, 2024)
print(summary_df.valid.value_counts())
print(summary_df[~summary_df.valid][
    ["race", "team", "driver1", "driver2", "d1_laps", "d2_laps", "ratio"]
])

2023: 23 events
skipping 2023 round 17: no lap data
skipping 2023 round 18: no lap data
skipping 2023 round 19: no lap data
skipping 2023 round 20: no lap data
skipping 2023 round 21: no lap data
skipping 2023 round 22: no lap data
valid
True     133
False     21
Name: count, dtype: int64
     race             team driver1 driver2  d1_laps  d2_laps  ratio
2       1           Alpine     GAS     OCO       47       30   0.64
4       1          Ferrari     LEC     SAI       32       49   0.65
6       1          McLaren     NOR     PIA       36       10   0.28
13      2     Aston Martin     ALO     STR       45       12   0.27
19      2         Williams     SAR     ALB       44       19   0.43
26      3         Mercedes     HAM     RUS       44       11   0.25
28      3         Williams     SAR     ALB       41        3   0.07
29      4       Alfa Romeo     ZHO     BOT       29       39   0.74
30      4       AlphaTauri     DEV     TSU        8       42   0.19
52      6     Aston Martin    

Apparently, there appears to be no systematic bias int he exclusions - They're spread across different teams.

In [ ]:
summary_df = get_race_data(2024, 2025)
print(summary_df.valid.value_counts())

In [21]:
schedule = fastf1.get_event_schedule(2023)
print(len(schedule))

23


# Collecting Lap-Level Data

We now collect **individual lap times** for all valid teammate pairs, which we need for the mixed effects model. We also collect pit stop durations and tyre compound usage per stint for the pit stop and tyre analyses.

In [ ]:
def get_lap_level_data(start_year, end_year):
    """Collect individual lap times, pit stops, and tyre data for all teammate pairs."""
    all_laps = []
    all_pitstops = []

    for year in range(start_year, end_year):
        try:
            time.sleep(5)
            schedule = fastf1.get_event_schedule(year)
            races = schedule[schedule.EventFormat != "testing"]
            print(f"{year}: {len(races)} races")

            for _, event in races.iterrows():
                rnd = event['RoundNumber']
                race_name = event['EventName']
                try:
                    time.sleep(5)
                    session = fastf1.get_session(year, rnd, "Race")
                    session.load()
                except Exception as e:
                    print(f"  Skipping {year} R{rnd} ({race_name}): {e}")
                    continue

                laps = session.laps
                clean = clean_data(laps)
                era = "2018-2021" if year <= 2021 else "2022-2024"

                teams = clean.groupby('Team')['Driver'].unique()
                for team, drivers in teams.items():
                    if len(drivers) != 2:
                        continue
                    d1, d2 = sorted(drivers)
                    d1_laps = clean[clean.Driver == d1]
                    d2_laps = clean[clean.Driver == d2]
                    n1, n2 = len(d1_laps), len(d2_laps)
                    if max(n1, n2) == 0:
                        continue
                    ratio = min(n1, n2) / max(n1, n2)
                    if ratio < 0.75:
                        continue

                    for _, lap in pd.concat([d1_laps, d2_laps]).iterrows():
                        lt = lap['LapTime']
                        if pd.isna(lt):
                            continue
                        all_laps.append({
                            "year": year,
                            "era": era,
                            "round": rnd,
                            "race_name": race_name,
                            "team": team,
                            "driver": lap['Driver'],
                            "teammate": d2 if lap['Driver'] == d1 else d1,
                            "pair": f"{d1}-{d2}",
                            "lap_number": lap['LapNumber'],
                            "lap_time_s": lt.total_seconds(),
                            "compound": lap['Compound'],
                            "tyre_life": lap['TyreLife'],
                            "stint": lap['Stint'],
                            "position": lap['Position'],
                        })

                # Pit stop data: compare pit durations for teammates
                for team, drivers in teams.items():
                    if len(drivers) != 2:
                        continue
                    d1, d2 = sorted(drivers)
                    for drv in [d1, d2]:
                        drv_laps = laps[laps.Driver == drv]
                        pit_in = drv_laps[drv_laps.PitInTime.notna()]
                        pit_out = drv_laps[drv_laps.PitOutTime.notna()]
                        for _, row in pit_in.iterrows():
                            out_row = pit_out[pit_out.LapNumber == row.LapNumber + 1]
                            if len(out_row) == 1:
                                pit_duration = (out_row.iloc[0].PitOutTime - row.PitInTime).total_seconds()
                                if 1 < pit_duration < 60:
                                    all_pitstops.append({
                                        "year": year,
                                        "era": era,
                                        "round": rnd,
                                        "race_name": race_name,
                                        "team": team,
                                        "driver": drv,
                                        "teammate": d2 if drv == d1 else d1,
                                        "pair": f"{d1}-{d2}",
                                        "pit_duration_s": pit_duration,
                                        "lap_number": row.LapNumber,
                                    })

                print(f"  {year} R{rnd} done ({race_name})")

        except Exception as e:
            print(f"{year}: FAILED - {e}")

    return pd.DataFrame(all_laps), pd.DataFrame(all_pitstops)

print("Function defined.")

In [ ]:
# Start with a small test (1 year) - expand to get_lap_level_data(2018, 2025) once confirmed working
laps_df, pits_df = get_lap_level_data(2024, 2025)
print(f"Laps collected: {len(laps_df)}")
print(f"Pit stops collected: {len(pits_df)}")
laps_df.head()

# Save/Load Data

To avoid re-downloading every time, we save the collected data to CSV.

In [ ]:
# Save
laps_df.to_csv("rq2_laps.csv", index=False)
pits_df.to_csv("rq2_pitstops.csv", index=False)
print("Saved.")

# To load later without re-downloading:
# laps_df = pd.read_csv("rq2_laps.csv")
# pits_df = pd.read_csv("rq2_pitstops.csv")

# Exploratory Visualisations

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Compute per-race average lap time for each driver, then teammate gap
race_avg = laps_df.groupby(['year', 'round', 'team', 'driver', 'pair'])['lap_time_s'].mean().reset_index()
race_avg.rename(columns={'lap_time_s': 'avg_lap_time'}, inplace=True)

# For each pair-race, compute the gap between the two drivers
gaps = []
for (year, rnd, team, pair), grp in race_avg.groupby(['year', 'round', 'team', 'pair']):
    if len(grp) != 2:
        continue
    d1_row, d2_row = grp.iloc[0], grp.iloc[1]
    gap = d1_row['avg_lap_time'] - d2_row['avg_lap_time']
    gaps.append({
        'year': year, 'round': rnd, 'team': team, 'pair': pair,
        'driver1': d1_row['driver'], 'driver2': d2_row['driver'],
        'gap_s': gap, 'abs_gap_s': abs(gap)
    })

gaps_df = pd.DataFrame(gaps)
print(f"Total race-level teammate comparisons: {len(gaps_df)}")
print(f"Mean absolute gap: {gaps_df.abs_gap_s.mean():.3f}s")
print(f"Median absolute gap: {gaps_df.abs_gap_s.median():.3f}s")

In [ ]:
# Distribution of teammate gaps
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].hist(gaps_df['gap_s'], bins=40, edgecolor='black', alpha=0.7)
axes[0].axvline(0, color='red', linestyle='--')
axes[0].set_xlabel("Teammate Gap (s) [Driver1 - Driver2]")
axes[0].set_ylabel("Count")
axes[0].set_title("Distribution of Teammate Lap Time Gaps")

sns.boxplot(data=gaps_df, x='team', y='abs_gap_s', ax=axes[1])
axes[1].set_xticklabels(axes[1].get_xticklabels(), rotation=45, ha='right')
axes[1].set_ylabel("Absolute Gap (s)")
axes[1].set_title("Teammate Gap by Team")

plt.tight_layout()
plt.show()

# Analysis 1: Linear Mixed Effects Model

We fit a linear mixed effects model to test whether there are systematic lap time differences between teammates after controlling for:
- **Tyre compound** (SOFT/MEDIUM/HARD)
- **Tyre life** (laps on current tyre)
- **Stint number**
- **Track position**

Random effects:
- **Driver** nested in **Team** (driver baseline skill within the same car)
- **Race** (circuit-specific variation)

The key coefficient is the **driver indicator within each team**: if one driver is systematically faster after controlling for confounders, this suggests either skill differences or differential resource allocation.

In [ ]:
import statsmodels.formula.api as smf

# Prepare data for the mixed model
model_df = laps_df.copy()
model_df = model_df.dropna(subset=['lap_time_s', 'compound', 'tyre_life', 'stint', 'position'])
model_df['compound'] = model_df['compound'].astype('category')
model_df['team_race'] = model_df['team'] + "_" + model_df['year'].astype(str) + "_R" + model_df['round'].astype(str)

# Create a "driver is first alphabetically in pair" indicator to give a consistent direction
model_df['is_driver1'] = (model_df['driver'] < model_df['teammate']).astype(int)

print(f"Model data: {len(model_df)} laps, {model_df.team.nunique()} teams, {model_df.pair.nunique()} pairs")
print(f"Compounds: {model_df.compound.value_counts().to_dict()}")

In [ ]:
# Fit the linear mixed effects model
# Fixed effects: tyre compound, tyre life, stint, position, driver indicator
# Random effects: driver nested in team, race
lme_model = smf.mixedlm(
    "lap_time_s ~ C(compound) + tyre_life + stint + position + is_driver1",
    data=model_df,
    groups=model_df["team_race"],
    re_formula="~is_driver1"
)
lme_result = lme_model.fit(reml=True)
print(lme_result.summary())

# Per-Team Teammate Analysis

We also fit a separate model per team to see which teams show the largest systematic gap between drivers.

In [ ]:
from scipy import stats

# Per-team: paired comparison of average race lap times
team_results = []
for team in gaps_df['team'].unique():
    team_gaps = gaps_df[gaps_df.team == team]
    if len(team_gaps) < 3:
        continue
    t_stat, p_val = stats.ttest_1samp(team_gaps['gap_s'], 0)
    team_results.append({
        'team': team,
        'n_races': len(team_gaps),
        'mean_gap_s': team_gaps['gap_s'].mean(),
        'std_gap_s': team_gaps['gap_s'].std(),
        'mean_abs_gap_s': team_gaps['abs_gap_s'].mean(),
        't_stat': t_stat,
        'p_value': p_val,
        'significant': p_val < 0.05
    })

team_results_df = pd.DataFrame(team_results).sort_values('p_value')
print(team_results_df.to_string(index=False))

In [ ]:
# Visualise per-team gaps
fig, ax = plt.subplots(figsize=(12, 6))
order = team_results_df.sort_values('mean_gap_s')['team']
sns.boxplot(data=gaps_df, x='team', y='gap_s', order=order, ax=ax)
ax.axhline(0, color='red', linestyle='--', alpha=0.7)
ax.set_xticklabels(ax.get_xticklabels(), rotation=45, ha='right')
ax.set_ylabel("Teammate Gap (s) [Driver1 - Driver2]")
ax.set_title("Teammate Lap Time Gap by Team (negative = Driver1 faster)")
plt.tight_layout()
plt.show()

# Analysis 2: Pit Stop Duration - Paired t-tests

We compare pit stop durations between teammates within the same race using paired t-tests. If one driver consistently receives faster pit stops, it may indicate preferential treatment.

In [ ]:
# Average pit stop duration per driver per race
pit_avg = pits_df.groupby(['year', 'round', 'team', 'driver', 'pair'])['pit_duration_s'].mean().reset_index()

# Build paired comparisons
pit_gaps = []
for (year, rnd, team, pair), grp in pit_avg.groupby(['year', 'round', 'team', 'pair']):
    if len(grp) != 2:
        continue
    d1_row, d2_row = grp.iloc[0], grp.iloc[1]
    pit_gaps.append({
        'year': year, 'round': rnd, 'team': team, 'pair': pair,
        'driver1': d1_row['driver'], 'driver2': d2_row['driver'],
        'pit_gap_s': d1_row['pit_duration_s'] - d2_row['pit_duration_s'],
        'abs_pit_gap_s': abs(d1_row['pit_duration_s'] - d2_row['pit_duration_s'])
    })

pit_gaps_df = pd.DataFrame(pit_gaps)
print(f"Pit stop comparisons: {len(pit_gaps_df)}")
print(f"Mean absolute pit gap: {pit_gaps_df.abs_pit_gap_s.mean():.3f}s")

# Overall: is there a systematic bias?
t_stat, p_val = stats.ttest_1samp(pit_gaps_df['pit_gap_s'], 0)
print(f"\nOverall paired t-test: t={t_stat:.3f}, p={p_val:.4f}")

In [ ]:
# Per-team pit stop analysis
pit_team_results = []
for team in pit_gaps_df['team'].unique():
    team_pits = pit_gaps_df[pit_gaps_df.team == team]
    if len(team_pits) < 3:
        continue
    t_stat, p_val = stats.ttest_1samp(team_pits['pit_gap_s'], 0)
    pit_team_results.append({
        'team': team,
        'n_races': len(team_pits),
        'mean_pit_gap_s': team_pits['pit_gap_s'].mean(),
        'std_pit_gap_s': team_pits['pit_gap_s'].std(),
        't_stat': t_stat,
        'p_value': p_val,
        'significant': p_val < 0.05
    })

pit_team_df = pd.DataFrame(pit_team_results).sort_values('p_value')
print(pit_team_df.to_string(index=False))

In [ ]:
# Pit stop gap visualisation
fig, ax = plt.subplots(figsize=(12, 6))
order = pit_team_df.sort_values('mean_pit_gap_s')['team']
sns.boxplot(data=pit_gaps_df, x='team', y='pit_gap_s', order=order, ax=ax)
ax.axhline(0, color='red', linestyle='--', alpha=0.7)
ax.set_xticklabels(ax.get_xticklabels(), rotation=45, ha='right')
ax.set_ylabel("Pit Stop Gap (s) [Driver1 - Driver2]")
ax.set_title("Teammate Pit Stop Duration Gap by Team")
plt.tight_layout()
plt.show()

# Analysis 3: Tyre Allocation - Chi-Square Tests

We test whether one driver within a team gets fresh tyres more often than the other using chi-square tests. If teams systematically favour one driver with better tyre strategies, we'd expect unequal fresh tyre allocation.

In [ ]:
# Count soft tyre stints per driver per team-season
# A "soft stint" is valuable because it's faster; getting more of them = strategic advantage
soft_counts = laps_df[laps_df.compound == 'SOFT'].groupby(
    ['year', 'team', 'driver', 'pair']
).agg(soft_laps=('lap_number', 'count')).reset_index()

total_counts = laps_df.groupby(
    ['year', 'team', 'driver', 'pair']
).agg(total_laps=('lap_number', 'count')).reset_index()

tyre_df = total_counts.merge(soft_counts, on=['year', 'team', 'driver', 'pair'], how='left')
tyre_df['soft_laps'] = tyre_df['soft_laps'].fillna(0).astype(int)
tyre_df['soft_pct'] = tyre_df['soft_laps'] / tyre_df['total_laps'] * 100

# Chi-square per team: are soft laps equally distributed between teammates?
tyre_results = []
for team in tyre_df['team'].unique():
    team_data = tyre_df[tyre_df.team == team]
    if len(team_data) < 2:
        continue
    drivers = team_data['driver'].unique()
    if len(drivers) != 2:
        continue
    d1_soft = team_data[team_data.driver == drivers[0]]['soft_laps'].sum()
    d2_soft = team_data[team_data.driver == drivers[1]]['soft_laps'].sum()
    d1_other = team_data[team_data.driver == drivers[0]]['total_laps'].sum() - d1_soft
    d2_other = team_data[team_data.driver == drivers[1]]['total_laps'].sum() - d2_soft
    
    if d1_soft + d2_soft == 0:
        continue
    
    contingency = np.array([[d1_soft, d1_other], [d2_soft, d2_other]])
    chi2, p_val, dof, expected = stats.chi2_contingency(contingency)
    
    tyre_results.append({
        'team': team,
        'driver1': drivers[0],
        'driver2': drivers[1],
        'd1_soft_laps': d1_soft,
        'd2_soft_laps': d2_soft,
        'd1_soft_pct': round(d1_soft / (d1_soft + d1_other) * 100, 1),
        'd2_soft_pct': round(d2_soft / (d2_soft + d2_other) * 100, 1),
        'chi2': chi2,
        'p_value': p_val,
        'significant': p_val < 0.05
    })

tyre_results_df = pd.DataFrame(tyre_results).sort_values('p_value')
print(tyre_results_df.to_string(index=False))

In [ ]:
# Tyre allocation visualisation
fig, ax = plt.subplots(figsize=(12, 6))
x = np.arange(len(tyre_results_df))
width = 0.35
ax.bar(x - width/2, tyre_results_df['d1_soft_pct'], width, label='Driver 1', alpha=0.8)
ax.bar(x + width/2, tyre_results_df['d2_soft_pct'], width, label='Driver 2', alpha=0.8)
ax.set_xticks(x)
ax.set_xticklabels(tyre_results_df['team'], rotation=45, ha='right')
ax.set_ylabel("Soft Tyre Usage (%)")
ax.set_title("Soft Tyre Allocation Between Teammates")
ax.legend()

# Mark significant differences
for i, row in enumerate(tyre_results_df.itertuples()):
    if row.significant:
        ax.text(i, max(row.d1_soft_pct, row.d2_soft_pct) + 1, '*', ha='center', fontsize=14, color='red')

plt.tight_layout()
plt.show()

# Summary

| Analysis | Method | What it tests |
|----------|--------|---------------|
| Lap time gaps | Linear Mixed Effects Model | Systematic speed differences between teammates controlling for tyre, stint, position |
| Pit stop gaps | Paired t-tests per team | Whether one driver consistently gets faster pit stops |
| Tyre allocation | Chi-square tests per team | Whether soft tyre usage is equally distributed between teammates |

Significant results in any of these suggest differential treatment or systematic performance differences within teams.